Exercise 1

In [8]:
import math, random, time, statistics
from MP2_exercise import estimate_pi_serial

if __name__ == '__main__':
    random.seed(42)
    num_samples = 10000000
    times = []
    pies = []
    for _ in range(3):
        t0 = time.perf_counter()
        pi_estimate = estimate_pi_serial(num_samples)
        times.append(time.perf_counter() - t0)
        pies.append(pi_estimate)
        print(f"Run {_+1}:")
        print(f"pi estimate: {pi_estimate:.6f}")
        print(f"error: {abs(pi_estimate - math.pi):.6f}")
        print(f"Time: {time.perf_counter() - t0:.3f} s")

    t_serial = statistics.median(times)
    print("averages:")
    print(f"pi estimate: {statistics.median(pies):.6f} (error: {abs(statistics.median(pies)-math.pi):.6f})")
    print(f"Serial time: {t_serial:.3f}s")

Run 1:
pi estimate: 3.142310
error: 0.000717
Time: 1.065 s
Run 2:
pi estimate: 3.141866
error: 0.000273
Time: 1.062 s
Run 3:
pi estimate: 3.141505
error: 0.000087
Time: 1.026 s
averages:
pi estimate: 3.141866 (error: 0.000273)
Serial time: 1.061s


How accurate is the estimate? On average: 0.000273 with 3 runs and 10kk samples  
Run several times — does it vary? yes  
What is the serial time? 1.036seconds This will be your speedup denominator in E3.

In [9]:
from MP2_exercise import estimate_pi_parallel
import os, time, statistics, random

if __name__ == '__main__':
    random.seed(42)
    num_samples = 10_000_000
    for num_proc in range(1, os.cpu_count() + 1):
        times = []
        for _ in range(3):
            t0 = time.perf_counter()
            pi_est = estimate_pi_parallel(num_samples, num_proc)
            times.append(time.perf_counter() - t0)
        t = statistics.median(times)
        print(f"{num_proc:2d} workers: {t:.3f}s pi={pi_est:.6f}")

 1 workers: 1.125s pi=3.142004
 2 workers: 0.571s pi=3.141253
 3 workers: 0.404s pi=3.141622
 4 workers: 0.319s pi=3.141145
 5 workers: 0.270s pi=3.141843
 6 workers: 0.239s pi=3.141242
 7 workers: 0.248s pi=3.141973
 8 workers: 0.253s pi=3.141893


Do all worker counts give the same ˆπ? Why or why not? Monte Carlo uses random sampling  
At which count do you first see a meaningful speedup? from 1 to 2 workers halves the runtume, afterwards the gain is decreasing

1. At which worker count p∗is speedup maximum?  
$$S_6=1.061/0.239=4.44$$

2. Does speedup plateau or drop beyond p∗? Why?   
yes, at S_5<. Probably parallel overhead - process spawning, inter-process communication, task distribution, result collection, scheduling costs. Further it aligns with Amdahl's law.

3. Back-solve implied serial fraction: s = 1/Sp∗−1/p∗ / 1−1/p∗ — what fraction of time is effectively serial (IPC overhead + spawning)?  
$$s = ((1/4.44)-(1/6))/1-(1/6)=0.07$$

4. Mac M1/M2/M3 users: do you see a slope change near worker 8 (E-cores)?  
Yes, apparently M chips use performance cores and efficiency cores